In [2]:
!pip install pgmpy

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.inference import VariableElimination
from pgmpy.sampling import BayesianModelSampling

model = DiscreteBayesianNetwork([
    ("heatwave", "overload"), ("heatwave", "powercut"),
    ("overload", "powercut"), ("powercut", "cancel")
])
sn = ["yes", "no"]

cpd_h = TabularCPD("heatwave", 2, [[0.3], [0.7]], state_names={"heatwave": sn})
cpd_a = TabularCPD("overload", 2, [[0.6, 0.1], [0.4, 0.9]],
                   evidence=["heatwave"], evidence_card=[2],
                   state_names={"overload": sn, "heatwave": sn})
cpd_c = TabularCPD("powercut", 2,
                   [[0.7, 0.2, 0.5, 0.05], [0.3, 0.8, 0.5, 0.95]],
                   evidence=["heatwave", "overload"], evidence_card=[2, 2],
                   state_names={"powercut": sn, "heatwave": sn, "overload": sn})
cpd_l = TabularCPD("cancel", 2, [[0.8, 0.1], [0.2, 0.9]],
                   evidence=["powercut"], evidence_card=[2],
                   state_names={"cancel": sn, "powercut": sn})

model.add_cpds(cpd_h, cpd_a, cpd_c, cpd_l)
print(model.check_model())          # True

# 2. Exact inference
infer = VariableElimination(model)
print(infer.query(["heatwave"], evidence={"cancel": "yes"}))   # yes ≈ 0.5367

# 3. Rejection sampling
samples = BayesianModelSampling(model).rejection_sample(
    evidence=[State("cancel", "yes")], size=10000)
print((samples["heatwave"] == "yes").mean())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pgmpy]


/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


True
+---------------+-----------------+
| heatwave      |   phi(heatwave) |
+===============+=================+
| heatwave(yes) |          0.5367 |
+---------------+-----------------+
| heatwave(no)  |          0.4633 |
+---------------+-----------------+


  0%|          | 0/10000 [00:00<?, ?it/s]

0.5436


The exact answer from variable elimination is 0.5367, and the rejection sampling estimate was 0.5436, which is very close. The small difference is due to random sampling error, and it would shrink with more samples.